# PubApp API Test Session

This notebook tests the live PubApp API used by the Unity app for Skills Practice 1 and Skills Practice 2. It does not load local Hugging Face models, use Riva, use NeMo guardrails directly, or drive the website UI.

The flow mirrors Unity at the API layer:

- `POST /v1/chat` with `target_agent="caregiver"` for parent responses.
- `POST /v1/chat` with `target_agent="coach"` for coach feedback/scoring.
- Optional `POST /v1/tts` with response text to verify the deployed text-to-speech contract.

Live calls are gated by `RUN_LIVE_API_TESTS`. The default dry-run mode validates payload construction and reporting without hitting production.

In [28]:
import json
import os
import re
import time
import uuid
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional

import pandas as pd
import requests

try:
    from IPython.display import Audio, display
except Exception:
    Audio = None
    display = print

    
    
FORCE_LIVE_API_TESTS = True
FORCE_TTS_SMOKE = True  # optional, for /v1/tts WAV checks


BASE_URL = os.getenv("PUBAPP_BASE_URL", "https://hpvcommunicationtraining.org").rstrip("/")
API_KEY = os.getenv("SPARC_API_KEY", "").strip()
REQUEST_TIMEOUT_SECONDS = float(os.getenv("PUBAPP_REQUEST_TIMEOUT_SECONDS", "90"))
REQUEST_RETRIES = int(os.getenv("PUBAPP_REQUEST_RETRIES", "1"))
RUN_LIVE_API_TESTS = 1
RUN_TTS_SMOKE = 1
TTS_VOICE = os.getenv("PUBAPP_TTS_VOICE", "").strip() or None
RESULTS_DIR = Path("pubapp_test_results")

pd.set_option("display.max_colwidth", 240)
pd.set_option("display.width", 220)

print(f"BASE_URL: {BASE_URL}")
print(f"RUN_LIVE_API_TESTS: {RUN_LIVE_API_TESTS}")
print(f"RUN_TTS_SMOKE: {RUN_TTS_SMOKE}")
print(f"API key configured: {bool(API_KEY)}")

BASE_URL: https://hpvcommunicationtraining.org
RUN_LIVE_API_TESTS: 1
RUN_TTS_SMOKE: 1
API key configured: False


## PubApp Client

The chat helper follows the contract in `v2/main.py`: chat returns text and metadata, while audio is requested separately with `/v1/tts`. `X-API-Key` is included only when `SPARC_API_KEY` is set.

In [29]:
class PubAppClient:
    def __init__(self, base_url: str, api_key: str = "", timeout: float = 90.0, retries: int = 1):
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.retries = max(0, retries)
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": "sparcp-pubapp-api-notebook/1.0"})
        if api_key:
            self.session.headers.update({"X-API-Key": api_key})

    def _url(self, path: str) -> str:
        return f"{self.base_url}/{path.lstrip('/')}"

    def _request(self, method: str, path: str, **kwargs) -> requests.Response:
        last_error = None
        for attempt in range(self.retries + 1):
            try:
                response = self.session.request(method, self._url(path), timeout=self.timeout, **kwargs)
                if response.status_code < 500 or attempt == self.retries:
                    return response
            except requests.RequestException as exc:
                last_error = exc
                if attempt == self.retries:
                    raise
            time.sleep(min(2 ** attempt, 5))
        raise RuntimeError(f"request failed after retries: {last_error}")

    def health(self) -> Dict[str, Any]:
        started = time.perf_counter()
        try:
            response = self._request("GET", "/health")
            latency_ms = round((time.perf_counter() - started) * 1000, 1)
            try:
                body = response.json()
            except ValueError:
                body = {"raw_text": response.text[:1000]}
            return {"ok": response.ok, "status_code": response.status_code, "latency_ms": latency_ms, "body": body}
        except Exception as exc:
            latency_ms = round((time.perf_counter() - started) * 1000, 1)
            return {"ok": False, "status_code": None, "latency_ms": latency_ms, "error": repr(exc), "body": None}

    def chat(self, payload: Dict[str, Any]) -> Dict[str, Any]:
        started = time.perf_counter()
        response = self._request("POST", "/v1/chat", json=payload)
        latency_ms = round((time.perf_counter() - started) * 1000, 1)
        try:
            body = response.json()
        except ValueError:
            body = {"raw_text": response.text[:4000]}
        return {"ok": response.ok, "status_code": response.status_code, "latency_ms": latency_ms, "body": body, "payload": payload}

    def tts(self, session_id: str, text: str, voice: Optional[str] = None) -> Dict[str, Any]:
        payload = {"session_id": session_id, "text": text}
        if voice:
            payload["voice"] = voice
        started = time.perf_counter()
        response = self._request("POST", "/v1/tts", json=payload, headers={"Accept": "audio/wav"})
        latency_ms = round((time.perf_counter() - started) * 1000, 1)
        return {
            "ok": response.ok,
            "status_code": response.status_code,
            "latency_ms": latency_ms,
            "content_type": response.headers.get("Content-Type", ""),
            "bytes": response.content if response.ok else b"",
            "text_preview": text[:160],
            "payload": payload,
        }

client = PubAppClient(BASE_URL, API_KEY, REQUEST_TIMEOUT_SECONDS, REQUEST_RETRIES)
health_result = client.health() if RUN_LIVE_API_TESTS else {
    "ok": None,
    "status_code": None,
    "latency_ms": None,
    "body": "Skipped because RUN_LIVE_API_TESTS is false.",
}
health_result

{'ok': True,
 'status_code': 200,
 'latency_ms': 46.2,
 'body': {'status': 'healthy',
  'models_loaded': True,
  'ready_for_traffic': True,
  'tts_backend': 'kokoro',
  'tts_ready': True,
  'riva_connected': False,
  'api_auth_enabled': False,
  'api_auth_configured': False,
  'api_contract_version': 'v1',
  'guardrails_enabled': False,
  'guardrails_loaded': True,
  'riva_client_pool_initialized': False,
  'firebase_creds_configured': True,
  'legacy_unity_compatibility': True,
  'websocket_path': '/ws/audio',
  'default_sample_rate_hz': 16000,
  'rag_loaded': True,
  'rag_embedding_model': 'sentence-transformers/all-mpnet-base-v2',
  'base_model_name': '/pubapps/jasondeanarnold/SPARCP/models/meta_llama/Llama3.1-8B-Instruct',
  'use_adapters': False,
  'adapters_loaded': []}}

## Scenario Fixtures

These fixtures are a compact, editable subset of the H5 caregiver and H6 coach notebooks. They keep the important behavior under test without reintroducing local model loading or Riva dependencies.

In [30]:
ANNE_SYSTEM_PROMPT = """<System_Prompt>
<Identity_and_Mission>You are Anne Palmer, a parent in the first SPARC-P skills practice session. You brought your 10 year old daughter Riley to her annual well-child visit. The clinician is practicing C-LEAR communication skills.</Identity_and_Mission>
<Primary_Directives>Stay in character as Anne Palmer. Respond only with what Anne would say. Do not describe actions, tones, or gestures. If the clinician goes off topic, briefly steer back to Riley's visit. Only respond in 1-2 sentences.</Primary_Directives>
<Persona>Character: Anne Palmer. Role: Biological mother. Child: Riley, 10 year old daughter, no major health problems. Background: Concerned about vaccines being given too early and wants to understand why HPV vaccine is recommended now.</Persona>
<Conversation_FOCUS>Primary concern: TOO YOUNG / SEX-RELATED CONCERNS. Real reason: Riley is not having sex yet, so why is it needed? Dialogue style: Polite, hesitant, easily overwhelmed by too much detail. First turn: express general hesitation without revealing the full concern. Second turn: share that Riley is young and not sexually active, especially if the clinician used a listening skill. Third turn: decide. Accept only if the clinician answers the timing/sex concern and gives a strong recommendation; otherwise decline.</Conversation_FOCUS>
</System_Prompt>"""

MAYA_SYSTEM_PROMPT = """<System_Prompt>
<Identity_and_Mission>You are Maya Pena, a parent in the second SPARC-P skills practice session. You brought your 9 year old daughter Luna to her annual well-child visit. The clinician is practicing C-LEAR communication skills.</Identity_and_Mission>
<Primary_Directives>Stay in character as Maya Pena. Respond only with what Maya would say. Do not describe actions, tones, or gestures. Wait for the clinician to speak first. If the clinician goes off topic, briefly steer back to Luna's visit. Keep responses to 1-2 sentences.</Primary_Directives>
<Persona>Character: Maya Pena. Role: Biological mother. Child: Luna, 9 year old daughter. Background: Generally open to vaccines, but worried by community stories about long-term side effects.</Persona>
<Conversation_FOCUS>Primary concern: SIDE EFFECTS / INFERTILITY. Real reason: Maya has heard the HPV vaccine can cause infertility and worries it could affect Luna's future ability to have children. Dialogue style: Warm, polite, cautious, and guarded until reassured.</Conversation_FOCUS>
</System_Prompt>"""

PHASE_PROMPTS = {
    "COUNSEL": "Respond to the clinician's opening HPV vaccine counsel statement. If this is early in the conversation, express hesitation without revealing the full underlying concern.",
    "LISTEN": "Respond to the clinician's listening or exploration attempt. Share more of the underlying concern if the clinician invited it respectfully.",
    "EMPATHIZE": "Respond to the clinician's empathy statement. Indicate whether you feel heard, then leave room for the clinician to answer your concern.",
    "ANSWER_RECOMMEND": "Respond to the clinician's answer and recommendation. Decide whether to accept or decline based on whether the concern was answered and a strong recommendation was given.",
    "FINAL": "Keep the final response brief and allow the conversation to end.",
}

SKILLS_PRACTICE_SCENARIOS = [
    {
        "id": "sp1_anne_riley",
        "training_type": "SkillsPractice1",
        "parent_key": "anne_palmer",
        "parent_name": "Anne Palmer",
        "child_name": "Riley",
        "system_prompt": ANNE_SYSTEM_PROMPT,
        "turns": [
            {"phase": "COUNSEL", "clinician": "Riley looks healthy today. She is due for the HPV vaccine, which helps prevent six types of cancer. I recommend she get it today and return in 6 to 12 months for the second dose.", "expected_parent_keywords": ["young", "need", "sure"], "coach_fixture": "counsel_complete"},
            {"phase": "LISTEN", "clinician": "Can you tell me more about what worries you about the HPV vaccine?", "expected_parent_keywords": ["young", "sex", "needed"], "coach_fixture": "listen_explore"},
            {"phase": "EMPATHIZE", "clinician": "That makes sense. Other parents have wondered about this too, and I can understand wanting to be careful while Riley is still young.", "expected_parent_keywords": ["young", "sex", "worry"], "coach_fixture": "empathy_normalize"},
            {"phase": "ANSWER_RECOMMEND", "clinician": "We give the HPV vaccine now because it works best at younger ages and protects Riley long before any future exposure. I strongly recommend she receive it today.", "expected_parent_keywords": ["okay", "sense", "get"], "coach_fixture": "answer_recommend_strong"},
        ],
    },
    {
        "id": "sp2_maya_luna",
        "training_type": "SkillsPractice2",
        "parent_key": "maya_pena",
        "parent_name": "Maya Pena",
        "child_name": "Luna",
        "system_prompt": MAYA_SYSTEM_PROMPT,
        "turns": [
            {"phase": "COUNSEL", "clinician": "Luna is due for the HPV vaccine. It is safe, recommended at her age, and helps prevent six types of cancer. I recommend she receive it today and come back for the second dose.", "expected_parent_keywords": ["side", "effects", "heard"], "coach_fixture": "counsel_complete"},
            {"phase": "LISTEN", "clinician": "It sounds like you have heard some things that worry you. Can you tell me more about what you have heard?", "expected_parent_keywords": ["infertility", "children", "future"], "coach_fixture": "listen_restate_explore"},
            {"phase": "EMPATHIZE", "clinician": "I understand why that would feel scary. Other parents have asked about infertility too, and it makes sense to want to protect Luna's future.", "expected_parent_keywords": ["stories", "believe", "worried"], "coach_fixture": "empathy_validate"},
            {"phase": "ANSWER_RECOMMEND", "clinician": "There is no evidence that the HPV vaccine causes infertility. It has been studied carefully, helps prevent cancer, and I strongly recommend Luna get it today.", "expected_parent_keywords": ["better", "okay", "feel"], "coach_fixture": "answer_recommend_infertility"},
        ],
    },
]

COACH_SYSTEM_PROMPT = """You are the C-LEAR Coach, an expert AI evaluator for the SPARC-P clinical communication simulation. Give concise, constructive feedback to the clinician. Use supportive tone, plain language, and specific examples. Do not speak to the parent persona. Do not mention numeric scores in learner-facing prose unless the prompt explicitly requests strict JSON for backend validation."""

COACH_FORMATIVE_FIXTURES = {
    "counsel_complete": {"phase": "COUNSEL", "expected_keywords": ["cancer", "recommend", "second dose", "safe"], "rubric": "Counsel should mention age/recommendation, cancer prevention, safety, and the second dose."},
    "listen_explore": {"phase": "LISTEN", "expected_keywords": ["listen", "explore", "concern"], "rubric": "Listen should invite the parent to share more or restate the concern without judgment."},
    "listen_restate_explore": {"phase": "LISTEN", "expected_keywords": ["listen", "explore", "concern"], "rubric": "Listen may include restating the concern and inviting more detail."},
    "empathy_normalize": {"phase": "EMPATHIZE", "expected_keywords": ["empathy", "normalize", "understand"], "rubric": "Empathy should acknowledge, normalize, or validate before answering."},
    "empathy_validate": {"phase": "EMPATHIZE", "expected_keywords": ["empathy", "validate", "understand"], "rubric": "Empathy should validate the worry and prepare for a focused answer."},
    "answer_recommend_strong": {"phase": "ANSWER_RECOMMEND", "expected_keywords": ["answer", "recommend", "strong"], "rubric": "Answer-Recommend should answer the age/timing concern and include a strong recommendation."},
    "answer_recommend_infertility": {"phase": "ANSWER_RECOMMEND", "expected_keywords": ["answer", "recommend", "infertility"], "rubric": "Answer-Recommend should directly address infertility concerns and include a strong recommendation."},
}

print(f"Loaded {len(SKILLS_PRACTICE_SCENARIOS)} practice scenarios and {len(COACH_FORMATIVE_FIXTURES)} coach fixtures.")

Loaded 2 practice scenarios and 7 coach fixtures.


## Unity-Style Session and Payload Assembly

Unity keeps a rolling conversation history, sends only the last user message as `user_message`, and includes a serialized message array when PubApp extended context is enabled. This notebook mirrors that shape and caps history to the last six messages like `ConversationSession.BuildRequestMessages`.

In [31]:
@dataclass
class ChatMessage:
    role: str
    content: str

@dataclass
class TranscriptEntry:
    speaker: str
    content: str
    mode: str
    phase: str
    timestampUtc: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

@dataclass
class UnityConversationSession:
    session_id: str
    training_type: str
    max_history: int = 6
    history: List[ChatMessage] = field(default_factory=list)
    transcript: List[TranscriptEntry] = field(default_factory=list)

    def append_user(self, text: str, phase: str) -> None:
        text = text.strip()
        self.history.append(ChatMessage("user", text))
        self.transcript.append(TranscriptEntry("user", text, self.training_type, phase))

    def append_assistant(self, text: str, phase: str) -> None:
        text = text.strip()
        if not text:
            return
        self.history.append(ChatMessage("assistant", text))
        self.transcript.append(TranscriptEntry("assistant", text, self.training_type, phase))

    def build_request_messages(self, system_prompt: str) -> List[ChatMessage]:
        if not system_prompt.strip():
            raise ValueError("system_prompt is required")
        return [ChatMessage("system", system_prompt.strip()), *self.history[-self.max_history:]]

    def transcript_text(self, mode_filter: Optional[str] = None) -> str:
        lines = []
        for entry in self.transcript:
            if mode_filter and entry.mode != mode_filter:
                continue
            if entry.speaker in {"user", "assistant"}:
                lines.append(f"{entry.speaker}: {entry.content}")
        return "\n".join(lines)

def new_session_id(prefix: str) -> str:
    safe_prefix = re.sub(r"[^a-zA-Z0-9_-]", "-", prefix).strip("-")[:32]
    return f"{safe_prefix}-{uuid.uuid4().hex[:12]}"

def build_system_prompt(base_prompt: str, phase_prompt: str) -> str:
    return f"{base_prompt.strip()}<System Prompt2>{phase_prompt.strip()} </System Prompt2>"

def build_phase_context(phase: str, training_type: str, phase_prompt: str) -> str:
    return f"phase_type={phase}; training_type={training_type}; parent_phase_prompt={phase_prompt.strip()}"

def messages_json(messages: Iterable[ChatMessage]) -> str:
    return json.dumps({"messages": [asdict(message) for message in messages]}, ensure_ascii=False)

def normalize_pubapp_text(body: Dict[str, Any]) -> str:
    return (body.get("caregiver_text") or body.get("response_text") or "").strip()

def build_chat_payload(*, session: UnityConversationSession, user_message: str, target_agent: str, system_prompt: str, phase_context: str, messages: List[ChatMessage]) -> Dict[str, Any]:
    return {
        "session_id": session.session_id,
        "user_message": user_message,
        "user_transcript": user_message,
        "system_prompt": system_prompt.strip() or None,
        "phase_context": phase_context.strip() or None,
        "message_history_json": messages_json(messages),
        "target_agent": target_agent,
        "include_legacy_audio_b64": False,
    }

def build_coach_payload(session_id: str, user_prompt: str, system_prompt: str, phase_context: str) -> Dict[str, Any]:
    messages = [ChatMessage("system", system_prompt.strip()), ChatMessage("user", user_prompt.strip())]
    return {
        "session_id": session_id,
        "user_message": user_prompt.strip(),
        "user_transcript": user_prompt.strip(),
        "system_prompt": system_prompt.strip(),
        "phase_context": phase_context.strip(),
        "message_history_json": messages_json(messages),
        "target_agent": "coach",
        "include_legacy_audio_b64": False,
    }

## API-Observed Grading Helpers

These checks focus on the deployed API behavior: HTTP status, response text, target agent, contract metadata, broad persona alignment, response concision, coach feedback shape, and optional WAV TTS smoke results. They do not require exact string matches to older local-model notebook outputs.

In [32]:
COACH_JSON_FIELDS = {"score", "summary", "evidence", "keepDoing", "tryNextTime", "notes"}

def lower_text(value: Any) -> str:
    return str(value or "").lower()

def keyword_hits(text: str, keywords: Iterable[str]) -> List[str]:
    low = lower_text(text)
    return [keyword for keyword in keywords if lower_text(keyword) in low]

def count_sentences(text: str) -> int:
    pieces = [piece for piece in re.split(r"[.!?]+\s*", text.strip()) if piece]
    return len(pieces)

def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None
    try:
        value = json.loads(text)
        return value if isinstance(value, dict) else None
    except json.JSONDecodeError:
        pass
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None
    try:
        value = json.loads(match.group(0))
        return value if isinstance(value, dict) else None
    except json.JSONDecodeError:
        return None

def grade_parent_response(row: Dict[str, Any]) -> Dict[str, Any]:
    text = row.get("response_text", "")
    hits = keyword_hits(text, row.get("expected_parent_keywords", []))
    meta = row.get("coach_feedback_meta") or {}
    return {
        "contract_ok": bool(row.get("http_ok")) and bool(text) and row.get("active_agent") in {"caregiver", "supervisor", None},
        "persona_ok": not any(term in lower_text(text) for term in ["as an ai", "chatbot", "simulation"]),
        "concise_ok": 0 < count_sentences(text) <= 3,
        "expected_keyword_hits": hits,
        "expected_keyword_hit_count": len(hits),
        "safe_meta": meta.get("safe", True),
    }

def grade_coach_response(row: Dict[str, Any]) -> Dict[str, Any]:
    text = row.get("response_text", "")
    hits = keyword_hits(text, row.get("expected_keywords", []))
    parsed = extract_json_object(text)
    json_field_hits = sorted(COACH_JSON_FIELDS.intersection(parsed.keys())) if parsed else []
    return {
        "contract_ok": bool(row.get("http_ok")) and bool(text) and row.get("active_agent") in {"coach", "supervisor", None},
        "supportive_ok": not any(term in lower_text(text) for term in ["incorrect", "failed", "score:", "0.5", "1.0"]),
        "expected_keyword_hits": hits,
        "expected_keyword_hit_count": len(hits),
        "json_parse_ok": parsed is not None,
        "json_field_hits": json_field_hits,
    }

def summarize_results(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    summary_cols = [col for col in ["http_ok", "contract_ok", "persona_ok", "concise_ok", "supportive_ok", "json_parse_ok"] if col in df.columns]
    return df.groupby(["scenario_id", "target_agent"], dropna=False)[summary_cols].mean(numeric_only=True).reset_index()

## Test Runners

Dry-run mode constructs the same payloads and records skipped HTTP results. Set `RUN_LIVE_API_TESTS=1` in the environment, or set `RUN_LIVE_API_TESTS = True` in the setup cell, to execute against the live PubApp instance. Set `RUN_TTS_SMOKE=1` to verify WAV bytes from `/v1/tts` after each caregiver response.

In [33]:
def maybe_call_tts(session_id: str, text: str) -> Dict[str, Any]:
    if not (RUN_LIVE_API_TESTS and RUN_TTS_SMOKE and text.strip()):
        return {"tts_ok": None, "tts_status_code": None, "tts_latency_ms": None, "tts_bytes": 0, "tts_content_type": None}
    result = client.tts(session_id, text, voice=TTS_VOICE)
    audio_bytes = result.get("bytes", b"") or b""
    return {
        "tts_ok": bool(result.get("ok")) and len(audio_bytes) > 44,
        "tts_status_code": result.get("status_code"),
        "tts_latency_ms": result.get("latency_ms"),
        "tts_bytes": len(audio_bytes),
        "tts_content_type": result.get("content_type"),
        "tts_audio_bytes": audio_bytes,
    }

def run_caregiver_turn(session: UnityConversationSession, scenario: Dict[str, Any], turn: Dict[str, Any]) -> Dict[str, Any]:
    phase = turn["phase"]
    clinician_text = turn["clinician"]
    phase_prompt = PHASE_PROMPTS[phase]
    system_prompt = build_system_prompt(scenario["system_prompt"], phase_prompt)

    session.append_user(clinician_text, phase)
    messages = session.build_request_messages(system_prompt)
    payload = build_chat_payload(
        session=session,
        user_message=clinician_text,
        target_agent="caregiver",
        system_prompt=system_prompt,
        phase_context=build_phase_context(phase, scenario["training_type"], phase_prompt),
        messages=messages,
    )

    if RUN_LIVE_API_TESTS:
        result = client.chat(payload)
        body = result.get("body") or {}
        response_text = normalize_pubapp_text(body)
        http_ok = result.get("ok")
        status_code = result.get("status_code")
        latency_ms = result.get("latency_ms")
    else:
        body = {"response_text": "", "caregiver_text": "", "active_agent": "caregiver", "api_contract_version": "dry-run"}
        response_text = ""
        http_ok = status_code = latency_ms = None

    session.append_assistant(response_text, phase)
    tts_result = maybe_call_tts(session.session_id, response_text)

    row = {
        "scenario_id": scenario["id"],
        "training_type": scenario["training_type"],
        "session_id": session.session_id,
        "target_agent": "caregiver",
        "phase": phase,
        "clinician_text": clinician_text,
        "response_text": response_text,
        "http_ok": http_ok,
        "status_code": status_code,
        "latency_ms": latency_ms,
        "active_agent": body.get("active_agent"),
        "api_contract_version": body.get("api_contract_version"),
        "coach_feedback_meta": body.get("coach_feedback_meta"),
        "expected_parent_keywords": turn.get("expected_parent_keywords", []),
        "payload": payload,
        **{k: v for k, v in tts_result.items() if k != "tts_audio_bytes"},
    }
    row.update(grade_parent_response(row))
    if "tts_audio_bytes" in tts_result:
        row["tts_audio_bytes"] = tts_result["tts_audio_bytes"]
    return row

def build_formative_coach_prompt(scenario: Dict[str, Any], turn: Dict[str, Any], transcript: str) -> str:
    fixture = COACH_FORMATIVE_FIXTURES[turn["coach_fixture"]]
    return "\n".join([
        "RETURN YOUR ANSWER AS CONCISE COACH FEEDBACK FOR THE CLINICIAN.",
        f"Practice Phase: {fixture['phase']}",
        f"Training Type: {scenario['training_type']}",
        f"Rubric: {fixture['rubric']}",
        "Do not mention numeric scores. Use Keep Doing and Try Next Time framing when useful.",
        "Here is the relevant transcript snippet for this phase:",
        transcript,
    ])

def build_final_coach_prompt(scenario: Dict[str, Any], transcript: str) -> str:
    return "\n".join([
        "Final Review: Review the transcript and give concise end-of-skills-practice feedback.",
        f"Training Type: {scenario['training_type']}",
        "Focus on Counsel, Listen, Empathize, and Answer-Recommend. For SkillsPractice2, use only this SP2 transcript.",
        "Do not mention numeric scores. Use supportive Keep Doing / Try Next Time language.",
        "Here is the relevant transcript snippet for this phase:",
        transcript,
    ])

def run_coach_check(session: UnityConversationSession, scenario: Dict[str, Any], turn: Dict[str, Any], final: bool = False) -> Dict[str, Any]:
    phase = "FINAL" if final else turn["phase"]
    transcript_filter = scenario["training_type"] if scenario["training_type"] == "SkillsPractice2" else None
    transcript = session.transcript_text(mode_filter=transcript_filter)
    user_prompt = build_final_coach_prompt(scenario, transcript) if final else build_formative_coach_prompt(scenario, turn, transcript)
    phase_context = build_phase_context(phase, scenario["training_type"], PHASE_PROMPTS.get(phase, "Final coach review"))
    coach_session_id = f"{session.session_id}-coach"
    payload = build_coach_payload(coach_session_id, user_prompt, COACH_SYSTEM_PROMPT, phase_context)

    if RUN_LIVE_API_TESTS:
        result = client.chat(payload)
        body = result.get("body") or {}
        response_text = normalize_pubapp_text(body)
        http_ok = result.get("ok")
        status_code = result.get("status_code")
        latency_ms = result.get("latency_ms")
    else:
        body = {"response_text": "", "caregiver_text": "", "active_agent": "coach", "api_contract_version": "dry-run"}
        response_text = ""
        http_ok = status_code = latency_ms = None

    fixture = COACH_FORMATIVE_FIXTURES.get(turn.get("coach_fixture"), {}) if not final else {}
    row = {
        "scenario_id": scenario["id"],
        "training_type": scenario["training_type"],
        "session_id": coach_session_id,
        "target_agent": "coach",
        "phase": phase,
        "clinician_text": user_prompt,
        "response_text": response_text,
        "http_ok": http_ok,
        "status_code": status_code,
        "latency_ms": latency_ms,
        "active_agent": body.get("active_agent"),
        "api_contract_version": body.get("api_contract_version"),
        "expected_keywords": fixture.get("expected_keywords", ["feedback", "practice"]),
        "payload": payload,
    }
    row.update(grade_coach_response(row))
    return row

def run_practice_scenario(scenario: Dict[str, Any]) -> List[Dict[str, Any]]:
    session = UnityConversationSession(session_id=new_session_id(scenario["id"]), training_type=scenario["training_type"])
    rows = []
    for turn in scenario["turns"]:
        rows.append(run_caregiver_turn(session, scenario, turn))
        if scenario["training_type"] == "SkillsPractice1":
            rows.append(run_coach_check(session, scenario, turn, final=False))
    rows.append(run_coach_check(session, scenario, scenario["turns"][-1], final=True))
    return rows

## Run the API-Level Session Tests

In dry-run mode, this cell saves constructed payloads and a report showing which HTTP/TTS checks were skipped. In live mode, it records status codes, latencies, normalized responses, grading checks, and optional WAV smoke results.

In [34]:
all_rows: List[Dict[str, Any]] = []
for scenario in SKILLS_PRACTICE_SCENARIOS:
    all_rows.extend(run_practice_scenario(scenario))

results_df = pd.DataFrame(all_rows)
export_df = results_df.drop(columns=["tts_audio_bytes"], errors="ignore").copy()
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
csv_path = RESULTS_DIR / f"pubapp_api_test_results_{timestamp}.csv"
json_path = RESULTS_DIR / f"pubapp_api_test_results_{timestamp}.json"
payload_path = RESULTS_DIR / f"pubapp_api_test_payloads_{timestamp}.json"

export_df.to_csv(csv_path, index=False)
export_df.drop(columns=["payload"], errors="ignore").to_json(json_path, orient="records", indent=2)
payload_path.write_text(json.dumps([row.get("payload") for row in all_rows], indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Rows: {len(results_df)}")
print(f"Results CSV: {csv_path}")
print(f"Results JSON: {json_path}")
print(f"Payload archive: {payload_path}")

display_columns = [
    "scenario_id", "training_type", "target_agent", "phase", "http_ok", "status_code", "latency_ms",
    "contract_ok", "persona_ok", "concise_ok", "supportive_ok", "json_parse_ok", "expected_keyword_hit_count",
    "tts_ok", "tts_status_code", "response_text",
]
display(results_df[[col for col in display_columns if col in results_df.columns]])

Rows: 14
Results CSV: pubapp_test_results/pubapp_api_test_results_20260429T201026Z.csv
Results JSON: pubapp_test_results/pubapp_api_test_results_20260429T201026Z.json
Payload archive: pubapp_test_results/pubapp_api_test_payloads_20260429T201026Z.json


,scenario_id,training_type,target_agent,phase,http_ok,status_code,latency_ms,contract_ok,persona_ok,concise_ok,supportive_ok,json_parse_ok,expected_keyword_hit_count,tts_ok,tts_status_code,response_text
0,sp1_anne_riley,SkillsPractice1,caregiver,COUNSEL,True,200,2930.8,True,True,True,NaN,NaN,1,True,200.0,"I'm not sure...I mean, I know it's good for the cancer prevention and all, but can we talk about this a little more?"
1,sp1_anne_riley,SkillsPractice1,coach,COUNSEL,True,200,7876.6,True,NaN,NaN,False,True,4,NaN,NaN,"{""score"": 0.5, ""summary"": ""The clinician starts to express hesitation, showing awareness of the vaccine's benefits, but could improve by directly addressing the patient's concerns and providing clear next steps."", ""evidence"": [""The clin..."
2,sp1_anne_riley,SkillsPractice1,caregiver,LISTEN,True,200,3205.3,True,True,False,NaN,NaN,1,True,200.0,"Well, it's just that Riley's so young, and she's not...um...you know, she's not involved with anyone like that yet, so I'm not sure why she needs it. I know it's good for later, but I'm just wondering if it's really necessary at this age."
3,sp1_anne_riley,SkillsPractice1,coach,LISTEN,True,200,9695.2,True,NaN,NaN,False,True,2,NaN,NaN,"{""score"": 0.5, ""summary"": ""The clinician partially demonstrated the skill of inviting the parent to share more or restate their concern without judgment. However, there is room for improvement."", ""evidence"": [""Inviting the parent to sha..."
4,sp1_anne_riley,SkillsPractice1,caregiver,EMPATHIZE,True,200,2817.6,True,True,True,NaN,NaN,0,True,200.0,"I feel like I'm being heard, but I still don't know if it's necessary for her right now. Can you tell me more about why they recommend giving it to 10-year-olds when they're not...you know, active yet?"
5,sp1_anne_riley,SkillsPractice1,coach,EMPATHIZE,True,200,8880.5,True,NaN,NaN,False,True,3,NaN,NaN,"{""score"": 0.5, ""summary"": ""The clinician shows some understanding and validation of the parent's concerns, but could improve by acknowledging or normalizing the parent's feelings more explicitly before answering."", ""evidence"": [""User: T..."
6,sp1_anne_riley,SkillsPractice1,caregiver,ANSWER_RECOMMEND,True,200,3774.0,True,True,True,NaN,NaN,2,True,200.0,"That makes sense, I guess. If it's best to give it at a younger age and it'll protect her even if she's not...you know, active for a while, then I suppose it's okay."
7,sp1_anne_riley,SkillsPractice1,coach,ANSWER_RECOMMEND,True,200,8436.8,True,NaN,NaN,False,True,2,NaN,NaN,"{""score"": 0.5, ""summary"": ""The clinician partially addressed the age/timing concern but could improve by more directly addressing the concern and providing a stronger recommendation."", ""evidence"": [""The clinician initially expressed unc..."
8,sp1_anne_riley,SkillsPractice1,coach,FINAL,True,200,10500.5,True,NaN,NaN,False,True,0,NaN,NaN,"{""score"": 0.5, ""summary"": ""The conversation demonstrates some elements of Counsel, Listen, and Empathize, but could benefit from more effective Answer-Recommendation and clarity in addressing concerns."", ""evidence"": [""Providing a clear ..."
9,sp2_maya_luna,SkillsPractice2,caregiver,COUNSEL,True,200,2796.2,True,True,True,NaN,NaN,1,True,200.0,"I'm not sure, doctor. I've just heard some concerns about the vaccine from friends, and I'm not really aware of what it involves."


## Summary

Use this summary to spot contract regressions quickly. In dry-run mode, HTTP-derived fields are intentionally blank because the notebook did not call production.

In [35]:
summary_df = summarize_results(results_df)
display(summary_df)

if not RUN_LIVE_API_TESTS:
    print("Dry run complete. Set RUN_LIVE_API_TESTS=1 to call the live PubApp APIs.")
elif RUN_TTS_SMOKE and "tts_audio_bytes" in results_df.columns and Audio is not None:
    first_audio = next((value for value in results_df["tts_audio_bytes"].dropna().tolist() if value), None)
    if first_audio:
        display(Audio(data=first_audio, autoplay=False))

,scenario_id,target_agent,http_ok,contract_ok
0,sp1_anne_riley,caregiver,1.0,1.0
1,sp1_anne_riley,coach,1.0,1.0
2,sp2_maya_luna,caregiver,1.0,1.0
3,sp2_maya_luna,coach,1.0,1.0


## Offline Result Grading

This section grades the captured PubApp responses after the live API run. It does not call the server again. Caregiver rows are graded against phase-specific scenario expectations, while coach rows are graded for valid C-LEAR score JSON and feedback quality.

In [ ]:
if "results_df" not in globals():
    result_files = sorted(RESULTS_DIR.glob("pubapp_api_test_results_*.csv"), key=lambda path: path.stat().st_mtime, reverse=True)
    if not result_files:
        raise FileNotFoundError("No pubapp_api_test_results_*.csv files found. Run the API test cells first.")
    latest_results_path = result_files[0]
    print(f"Loading latest saved results for offline grading: {latest_results_path}")
    results_df = pd.read_csv(latest_results_path)


def keyword_any(text: str, terms: Iterable[str]) -> bool:
    low = lower_text(text)
    return any(lower_text(term) in low for term in terms)

def keyword_count(text: str, terms: Iterable[str]) -> int:
    low = lower_text(text)
    return sum(1 for term in terms if lower_text(term) in low)

def score_from_checks(checks: Dict[str, bool]) -> float:
    if not checks:
        return 0.0
    return round(sum(1 for value in checks.values() if value) / len(checks), 2)

def letter_grade(score: Optional[float]) -> str:
    if score is None:
        return "SKIP"
    if score >= 0.9:
        return "A"
    if score >= 0.8:
        return "B"
    if score >= 0.7:
        return "C"
    if score >= 0.6:
        return "D"
    return "F"

PARENT_BEHAVIOR_RUBRICS = {
    ("sp1_anne_riley", "COUNSEL"): {
        "must": {"hesitation": ["not sure", "talk", "little more", "young", "need"]},
        "must_not": {"premature_deep_concern": ["sex", "active yet", "not involved"]},
    },
    ("sp1_anne_riley", "LISTEN"): {
        "must": {"too_young": ["young", "10"], "sex_related_concern": ["sex", "active", "involved"], "need_question": ["need", "necessary", "why"]},
        "must_not": {},
    },
    ("sp1_anne_riley", "EMPATHIZE"): {
        "must": {"feels_heard": ["heard", "listening", "understand"], "still_has_concern": ["still", "necessary", "why", "recommend"]},
        "must_not": {},
    },
    ("sp1_anne_riley", "ANSWER_RECOMMEND"): {
        "must": {"accepts_or_soft_accepts": ["okay", "ok", "makes sense", "suppose", "we can", "get it"], "reason_reflected": ["younger", "protect", "active", "exposure"]},
        "must_not": {"declines": ["no", "decline", "not today", "won't"]},
    },
    ("sp2_maya_luna", "COUNSEL"): {
        "must": {"hesitation": ["not sure", "concern", "heard", "friends", "worry"]},
        "must_not": {"premature_infertility": ["infertility"]},
    },
    ("sp2_maya_luna", "LISTEN"): {
        "must": {"infertility_concern": ["infertility"], "concern_affect": ["concern", "worry", "scary"]},
        "must_not": {},
    },
    ("sp2_maya_luna", "EMPATHIZE"): {
        "must": {"feels_heard": ["listening", "heard", "appreciate"], "asks_for_evidence": ["science", "evidence", "understand", "infertility"]},
        "must_not": {},
    },
    ("sp2_maya_luna", "ANSWER_RECOMMEND"): {
        "must": {"safety_reflected": ["safe", "no evidence", "protect", "cancer"], "decision_or_readiness": ["okay", "understand", "make sure", "get it", "today"]},
        "must_not": {"declines": ["no", "decline", "not today", "won't"]},
    },
}

def grade_parent_behavior(row: Dict[str, Any]) -> Dict[str, Any]:
    if row.get("target_agent") != "caregiver":
        return {}
    text = str(row.get("response_text") or "")
    rubric = PARENT_BEHAVIOR_RUBRICS.get((row.get("scenario_id"), row.get("phase")), {"must": {}, "must_not": {}})
    checks = {
        "http_success": bool(row.get("http_ok")),
        "non_empty": bool(text.strip()),
        "in_character": not keyword_any(text, ["as an ai", "chatbot", "simulation", "language model"]),
        "concise": 0 < count_sentences(text) <= 3,
    }
    for name, terms in rubric.get("must", {}).items():
        checks[name] = keyword_any(text, terms)
    for name, terms in rubric.get("must_not", {}).items():
        checks[f"no_{name}"] = not keyword_any(text, terms)
    score = score_from_checks(checks)
    failed = [name for name, ok in checks.items() if not ok]
    return {
        "local_behavior_score": score,
        "local_grade": letter_grade(score),
        "local_grade_reasons": "pass" if not failed else "; ".join(failed),
        "local_checks": checks,
    }

def grade_coach_behavior(row: Dict[str, Any]) -> Dict[str, Any]:
    if row.get("target_agent") != "coach":
        return {}
    text = str(row.get("response_text") or "")
    parsed = extract_json_object(text)
    checks = {
        "http_success": bool(row.get("http_ok")),
        "non_empty": bool(text.strip()),
        "valid_json": isinstance(parsed, dict),
    }
    if isinstance(parsed, dict):
        score_value = parsed.get("score")
        try:
            numeric_score = float(score_value)
        except (TypeError, ValueError):
            numeric_score = None
        checks.update({
            "score_allowed": numeric_score in {0.0, 0.5, 1.0},
            "has_summary": bool(str(parsed.get("summary") or "").strip()),
            "has_evidence": isinstance(parsed.get("evidence"), list) and len(parsed.get("evidence")) > 0,
            "has_keep_or_try": (isinstance(parsed.get("keepDoing"), list) and len(parsed.get("keepDoing")) > 0) or (isinstance(parsed.get("tryNextTime"), list) and len(parsed.get("tryNextTime")) > 0),
            "not_overlong": len(text) <= 1800,
        })
    else:
        numeric_score = None
        checks.update({"score_allowed": False, "has_summary": False, "has_evidence": False, "has_keep_or_try": False, "not_overlong": False})
    score = score_from_checks(checks)
    failed = [name for name, ok in checks.items() if not ok]
    return {
        "local_behavior_score": score,
        "local_grade": letter_grade(score),
        "coach_model_score": numeric_score,
        "local_grade_reasons": "pass" if not failed else "; ".join(failed),
        "local_checks": checks,
    }

def grade_result_row(row: pd.Series) -> Dict[str, Any]:
    data = row.to_dict()
    if data.get("target_agent") == "caregiver":
        return grade_parent_behavior(data)
    if data.get("target_agent") == "coach":
        return grade_coach_behavior(data)
    return {"local_behavior_score": None, "local_grade": "SKIP", "local_grade_reasons": "unknown target_agent", "local_checks": {}}

def build_graded_results(df: pd.DataFrame) -> pd.DataFrame:
    grades = pd.DataFrame([grade_result_row(row) for _, row in df.iterrows()])
    graded = pd.concat([df.reset_index(drop=True), grades.reset_index(drop=True)], axis=1)
    return graded

graded_results_df = build_graded_results(results_df)
summary_grade_df = (
    graded_results_df
    .groupby(["scenario_id", "target_agent"], dropna=False)
    .agg(
        rows=("local_behavior_score", "count"),
        mean_local_behavior_score=("local_behavior_score", "mean"),
        min_local_behavior_score=("local_behavior_score", "min"),
        mean_latency_ms=("latency_ms", "mean"),
    )
    .reset_index()
)
summary_grade_df["mean_local_grade"] = summary_grade_df["mean_local_behavior_score"].apply(letter_grade)

display_columns = [
    "scenario_id", "target_agent", "phase", "local_behavior_score", "local_grade", "coach_model_score",
    "local_grade_reasons", "response_text",
]
display(graded_results_df[[col for col in display_columns if col in graded_results_df.columns]])
display(summary_grade_df)

In [ ]:
graded_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
graded_csv_path = RESULTS_DIR / f"pubapp_api_test_graded_results_{graded_timestamp}.csv"
graded_json_path = RESULTS_DIR / f"pubapp_api_test_graded_results_{graded_timestamp}.json"
grade_summary_path = RESULTS_DIR / f"pubapp_api_test_grade_summary_{graded_timestamp}.csv"

graded_export_df = graded_results_df.drop(columns=["payload", "tts_audio_bytes"], errors="ignore").copy()
graded_export_df.to_csv(graded_csv_path, index=False)
graded_export_df.to_json(graded_json_path, orient="records", indent=2)
summary_grade_df.to_csv(grade_summary_path, index=False)

print(f"Graded results CSV: {graded_csv_path}")
print(f"Graded results JSON: {graded_json_path}")
print(f"Grade summary CSV: {grade_summary_path}")